# XGBoost Tunado — Validação Cruzada e Métricas

Este notebook adapta o modelo para **XGBoost com tuning de hiperparâmetros** usando `RandomizedSearchCV`.

Fluxo:

1. Carregar a base `final_dataset.csv`.
2. Agrupar `Extreme rain` junto com `Rain`.
3. Transformar o alvo em binário: `Rain = 1` e `No rain = 0`.
4. Montar pipeline com `RandomUnderSampler`, `OneHotEncoder` e `XGBClassifier`.
5. Fazer tuning dos hiperparâmetros.
6. Avaliar o modelo tunado com validação cruzada estratificada.
7. Gerar matriz de confusão, curva ROC, curva Precision-Recall, tabela de métricas e feature importance.

In [1]:
# Se necessário, instale antes de rodar:
# import sys
# !{sys.executable} -m pip install xgboost imbalanced-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_curve,
    average_precision_score,
    PrecisionRecallDisplay,
    recall_score,
    precision_score,
    f1_score,
    make_scorer,
    classification_report
)

from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

In [2]:
inicio = time.perf_counter()

In [3]:
# Reprodutibilidade
SEED = 123

# Número de folds para avaliação final
N_SPLITS_AVALIACAO = 5

# Número de folds usados dentro do tuning.
# Use 3 para reduzir tempo; se quiser mais rigor, use 5.
N_SPLITS_TUNING = 3

# Número de combinações aleatórias de hiperparâmetros testadas.
# Para testar rápido: 10
# Para um tuning melhor: 30, 50 ou mais
N_ITER_TUNING = 5

In [4]:
# A base final_dataset.csv deve estar na mesma pasta do notebook.
df = pd.read_csv("final_dataset.csv")

df.head()

,radiation_1h,radiation_2h,radiation_3h,humidity_2h,humidity_3h,humidity_1h,temperature_2h,temperature_1h,temperature_3h,pressure_1h,...,pressure_2h,dp_temperature_1h,dp_temperature_2h,dp_temperature_3h,WindSpeed_1h,WindSpeed_2h,WindSpeed_3h,precipitation,season,dayperiod
0,Night/Zero,Night/Zero,Night/Zero,High,High,Moderate,Moderate/Low,Moderate/Low,Moderate/Low,High,...,High,Moderate,Moderate,Moderate,Low,Low,Low,No rain,autumn,afternoon
1,Night/Zero,Night/Zero,Night/Zero,Moderate,High,Moderate,Moderate/Low,Moderate,Moderate/Low,Moderate,...,High,High,Moderate,Moderate,Low,Low,Low,No rain,autumn,afternoon
2,Night/Zero,Night/Zero,Night/Zero,Moderate,Moderate,High,Moderate,Moderate/Low,Moderate/Low,Moderate,...,Moderate,Moderate,High,Moderate,High,Low,Low,No rain,autumn,afternoon
3,Night/Zero,Night/Zero,Night/Zero,High,Moderate,High,Moderate/Low,Moderate/Low,Moderate,Moderate,...,Moderate,High,Moderate,High,Low,Moderate,Low,No rain,autumn,night
4,Night/Zero,Night/Zero,Night/Zero,Moderate,High,High,Moderate/Low,Moderate/Low,Moderate/Low,Moderate,...,Moderate,Moderate,High,Moderate,Moderate,Low,Moderate,No rain,autumn,night


In [5]:
# Agrupa chuva intensa na classe Rain
df["precipitation"] = df["precipitation"].replace("Extreme rain", "Rain")

X = df.drop(columns=["precipitation"])

# Para o XGBoost, é mais seguro usar alvo numérico:
# Rain = 1
# No rain = 0
y = (df["precipitation"] == "Rain").astype(int)

# Ordem usada na matriz de confusão:
# labels=[1, 0]
# cm[0, 0] = Rain predito como Rain
# cm[0, 1] = Rain predito como No rain
# cm[1, 0] = No rain predito como Rain
# cm[1, 1] = No rain predito como No rain
labels = [1, 0]
labels_display = ["Rain", "No rain"]

print("Distribuição original das classes:")
display(df["precipitation"].value_counts())

print("Distribuição binária usada no modelo:")
display(y.value_counts().rename(index={1: "Rain", 0: "No rain"}))

Distribuição original das classes:


precipitation
No rain    431380
Rain        37431
Name: count, dtype: int64

Distribuição binária usada no modelo:


precipitation
No rain    431380
Rain        37431
Name: count, dtype: int64

## 1. Funções auxiliares

A função `metricas_da_matriz` considera `Rain` como classe positiva.

Assim:

- **Sensibilidade** = capacidade de detectar chuva.
- **Especificidade** = capacidade de reconhecer ausência de chuva.

In [6]:
def criar_pipeline_xgboost(seed=SEED, parametros_modelo=None):
    """Cria pipeline com undersampling, one-hot encoding e XGBoost.

    O undersampling fica dentro da validação cruzada, evitando vazamento de dados.
    """
    if parametros_modelo is None:
        parametros_modelo = {}

    parametros_padrao = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": seed,
        "n_jobs": -1,
        "tree_method": "hist"
    }
    parametros_padrao.update(parametros_modelo)

    return Pipeline(steps=[
        ("undersampling", RandomUnderSampler(random_state=seed)),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ("modelo", XGBClassifier(**parametros_padrao))
    ])


def metricas_da_matriz(cm):
    """Calcula métricas considerando a ordem: labels = [1, 0]."""
    VP = cm[0, 0]
    FN = cm[0, 1]
    FP = cm[1, 0]
    VN = cm[1, 1]

    acuracia = (VP + VN) / cm.sum()
    sensibilidade = VP / (VP + FN) if (VP + FN) > 0 else np.nan
    especificidade = VN / (VN + FP) if (VN + FP) > 0 else np.nan
    precisao = VP / (VP + FP) if (VP + FP) > 0 else np.nan
    f1 = (2 * precisao * sensibilidade / (precisao + sensibilidade)
          if (precisao + sensibilidade) > 0 else np.nan)
    balanced_acc = (sensibilidade + especificidade) / 2

    return {
        "Acurácia": acuracia,
        "Sensibilidade_Rain": sensibilidade,
        "Especificidade_No_rain": especificidade,
        "Precisão_Rain": precisao,
        "F1_Rain": f1,
        "Balanced_accuracy": balanced_acc
    }

## 2. Tuning de hiperparâmetros

Aqui o `RandomizedSearchCV` testa combinações de hiperparâmetros do XGBoost.

Como o XGBoost está dentro do `Pipeline` com o nome `modelo`, todos os parâmetros começam com:

```python
modelo__
```

A métrica principal usada para escolher o melhor modelo é `roc_auc`.

In [ ]:
cv_tuning = StratifiedKFold(
    n_splits=N_SPLITS_TUNING,
    shuffle=True,
    random_state=SEED
)

pipeline_base = criar_pipeline_xgboost(seed=SEED)

param_distributions = {
    "modelo__n_estimators": [100, 200, 300, 500],
    "modelo__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "modelo__max_depth": [2, 3, 4, 5, 6],
    "modelo__min_child_weight": [1, 3, 5, 10],
    "modelo__subsample": [0.6, 0.8, 1.0],
    "modelo__colsample_bytree": [0.6, 0.8, 1.0],
    "modelo__gamma": [0, 0.1, 0.5, 1],
    "modelo__reg_alpha": [0, 0.01, 0.1, 1],
    "modelo__reg_lambda": [1, 3, 5, 10]
}

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "sensibilidade_rain": make_scorer(recall_score, pos_label=1),
    "especificidade_no_rain": make_scorer(recall_score, pos_label=0),
    "precision_rain": make_scorer(precision_score, pos_label=1, zero_division=0),
    "f1_rain": make_scorer(f1_score, pos_label=1, zero_division=0)
}

busca_xgb = RandomizedSearchCV(
    estimator=pipeline_base,
    param_distributions=param_distributions,
    n_iter=N_ITER_TUNING,
    scoring=scoring,
    refit="roc_auc",
    cv=cv_tuning,
    verbose=2,
    random_state=SEED,
    n_jobs=1,
    return_train_score=True
)

busca_xgb.fit(X, y)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.1, modelo__learning_rate=0.01, modelo__max_depth=2, modelo__min_child_weight=3, modelo__n_estimators=500, modelo__reg_alpha=0.1, modelo__reg_lambda=3, modelo__subsample=1.0; total time=   1.8s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.1, modelo__learning_rate=0.01, modelo__max_depth=2, modelo__min_child_weight=3, modelo__n_estimators=500, modelo__reg_alpha=0.1, modelo__reg_lambda=3, modelo__subsample=1.0; total time=   1.7s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.1, modelo__learning_rate=0.01, modelo__max_depth=2, modelo__min_child_weight=3, modelo__n_estimators=500, modelo__reg_alpha=0.1, modelo__reg_lambda=3, modelo__subsample=1.0; total time=   1.7s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.1, modelo__learning_rate=0.1, modelo__max_depth=3, modelo__min_child_weight=3, modelo__n_estimators=500, modelo__reg_alpha=1, modelo__reg_lam

In [ ]:
print("Melhor AUC no tuning:", busca_xgb.best_score_)
print("\nMelhores hiperparâmetros:")

for param, valor in busca_xgb.best_params_.items():
    print(param, ":", valor)

resultados_tuning = pd.DataFrame(busca_xgb.cv_results_).sort_values(
    "rank_test_roc_auc"
)

colunas_tuning = [
    "rank_test_roc_auc",
    "mean_test_roc_auc",
    "std_test_roc_auc",
    "mean_test_average_precision",
    "mean_test_sensibilidade_rain",
    "mean_test_especificidade_no_rain",
    "mean_test_precision_rain",
    "mean_test_f1_rain",
    "mean_test_accuracy",
    "mean_test_balanced_accuracy",
    "params"
]

display(resultados_tuning[colunas_tuning].head(10))

resultados_tuning.to_csv("resultados_tuning_xgboost.csv", index=False)

## 3. Criação do modelo XGBoost tunado

A célula abaixo pega os melhores hiperparâmetros encontrados no tuning e cria uma função para usar esse modelo na avaliação final.

In [ ]:
melhores_parametros_xgb = {
    chave.replace("modelo__", ""): valor
    for chave, valor in busca_xgb.best_params_.items()
}

melhores_parametros_xgb

In [ ]:
def criar_modelo_xgboost_tunado(seed=SEED):
    """Cria o pipeline do XGBoost com os melhores hiperparâmetros encontrados."""
    return criar_pipeline_xgboost(
        seed=seed,
        parametros_modelo=melhores_parametros_xgb
    )

## 4. Avaliação final com validação cruzada

Agora o modelo tunado é avaliado com validação cruzada estratificada.

Nesta etapa são calculadas:

- Matriz de confusão acumulada.
- Acurácia.
- Sensibilidade para chuva.
- Especificidade para não chuva.
- Precisão para chuva.
- F1-score para chuva.
- Balanced accuracy.
- AUC ROC.
- Average Precision / PR-AUC.

In [ ]:
skf = StratifiedKFold(
    n_splits=N_SPLITS_AVALIACAO,
    shuffle=True,
    random_state=SEED
)

cm_total = np.zeros((2, 2), dtype=int)
matrizes_folds = {}
resultados = []

y_test_total = []
y_pred_total = []
y_prob_total = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    modelo_xgb_tunado = criar_modelo_xgboost_tunado()
    modelo_xgb_tunado.fit(X_train_fold, y_train_fold)

    y_pred_fold = modelo_xgb_tunado.predict(X_test_fold)
    y_prob_fold = modelo_xgb_tunado.predict_proba(X_test_fold)[:, 1]

    cm_fold = confusion_matrix(
        y_test_fold,
        y_pred_fold,
        labels=labels
    )

    cm_total += cm_fold
    matrizes_folds[f"Fold {fold}"] = cm_fold

    metricas = metricas_da_matriz(cm_fold)
    metricas["AUC_ROC"] = roc_auc_score(y_test_fold, y_prob_fold)
    metricas["Average_precision_PR_AUC"] = average_precision_score(y_test_fold, y_prob_fold)

    for nome_metrica, valor in metricas.items():
        resultados.append({
            "Fold": fold,
            "Métrica": nome_metrica,
            "Valor": valor
        })

    y_test_total.extend(y_test_fold)
    y_pred_total.extend(y_pred_fold)
    y_prob_total.extend(y_prob_fold)

    print(f"Fold {fold} concluído")

In [ ]:
matriz_confusao_total = pd.DataFrame(
    cm_total,
    index=["Real Rain", "Real No rain"],
    columns=["Predito Rain", "Predito No rain"]
)

resultados_cv = pd.DataFrame(resultados)

tabela_cv = resultados_cv.pivot(
    index="Métrica",
    columns="Fold",
    values="Valor"
)

tabela_cv.columns = [f"Fold {col}" for col in tabela_cv.columns]
tabela_cv["Média"] = tabela_cv.mean(axis=1)
tabela_cv["DP"] = tabela_cv.std(axis=1)

print("Matriz de confusão acumulada:")
display(matriz_confusao_total)

print("Métricas por fold:")
display(tabela_cv)

matriz_confusao_total.to_csv("matriz_confusao_xgboost_tunado.csv")
tabela_cv.to_csv("metricas_cv_xgboost_tunado.csv")

In [ ]:
# Relatório geral usando todas as predições dos folds
print(classification_report(
    y_test_total,
    y_pred_total,
    target_names=["No rain", "Rain"],
    labels=[0, 1],
    zero_division=0
))

## 5. Matriz de confusão

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm_total.astype(int),
    display_labels=labels_display
).plot(values_format="d")

plt.title("Matriz de Confusão - XGBoost Tunado - Validação Cruzada")
plt.xlabel("Classe predita")
plt.ylabel("Classe real")

plt.savefig("matriz_confusao_xgboost_tunado.png", dpi=300, bbox_inches="tight")

plt.show()

## 6. Curva ROC

In [ ]:
y_test_total = np.array(y_test_total)
y_pred_total = np.array(y_pred_total)
y_prob_total = np.array(y_prob_total)

fpr, tpr, thresholds = roc_curve(y_test_total, y_prob_total)
auc = roc_auc_score(y_test_total, y_prob_total)

print("AUC ROC geral:", auc)

RocCurveDisplay(
    fpr=fpr,
    tpr=tpr,
    roc_auc=auc,
    estimator_name="XGBoost Tunado"
).plot()

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Curva ROC - XGBoost Tunado - Validação Cruzada")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")

plt.savefig("curva_roc_xgboost_tunado.png", dpi=300, bbox_inches="tight")

plt.show()

## 7. Curva Precision-Recall

Essa curva é útil porque sua base é desbalanceada: existem muito mais casos de `No rain` do que de `Rain`.

In [ ]:
precision, recall, thresholds_pr = precision_recall_curve(y_test_total, y_prob_total)
average_precision = average_precision_score(y_test_total, y_prob_total)

print("Average Precision / PR-AUC geral:", average_precision)

PrecisionRecallDisplay(
    precision=precision,
    recall=recall,
    average_precision=average_precision,
    estimator_name="XGBoost Tunado"
).plot()

plt.title("Curva Precision-Recall - XGBoost Tunado - Validação Cruzada")
plt.xlabel("Recall / Sensibilidade")
plt.ylabel("Precisão")

plt.savefig("curva_precision_recall_xgboost_tunado.png", dpi=300, bbox_inches="tight")

plt.show()

## 8. Ajuste opcional do limiar de decisão

O XGBoost gera probabilidades. O padrão é classificar como `Rain` quando a probabilidade é maior ou igual a `0.5`.

Se você quiser aumentar a sensibilidade para chuva, pode reduzir o limiar. Isso tende a detectar mais chuvas, mas também aumenta falsos positivos.

In [ ]:
limiares = np.arange(0.10, 0.91, 0.05)
resultados_limiares = []

for limiar in limiares:
    y_pred_limiar = (y_prob_total >= limiar).astype(int)

    cm_limiar = confusion_matrix(
        y_test_total,
        y_pred_limiar,
        labels=labels
    )

    metricas = metricas_da_matriz(cm_limiar)
    metricas["Limiar"] = limiar
    resultados_limiares.append(metricas)

tabela_limiares = pd.DataFrame(resultados_limiares)

tabela_limiares = tabela_limiares[[
    "Limiar",
    "Acurácia",
    "Sensibilidade_Rain",
    "Especificidade_No_rain",
    "Precisão_Rain",
    "F1_Rain",
    "Balanced_accuracy"
]]

display(tabela_limiares)

tabela_limiares.to_csv("metricas_por_limiar_xgboost_tunado.csv", index=False)

## 9. Feature importance do modelo tunado

Agora treinamos o modelo tunado na base completa para extrair a importância das variáveis.

Atenção: como usamos `OneHotEncoder`, a importância aparece por categoria criada pelo encoder.

In [ ]:
modelo_final_xgb_tunado = criar_modelo_xgboost_tunado()
modelo_final_xgb_tunado.fit(X, y)

encoder = modelo_final_xgb_tunado.named_steps["encoder"]
modelo = modelo_final_xgb_tunado.named_steps["modelo"]

nomes_features = encoder.get_feature_names_out(X.columns)

importancia = pd.DataFrame({
    "Variável": nomes_features,
    "Importância": modelo.feature_importances_
}).sort_values("Importância", ascending=False)

display(importancia.head(30))

importancia.to_csv("feature_importance_xgboost_tunado.csv", index=False)

importancia.head(30).plot(
    x="Variável",
    y="Importância",
    kind="barh",
    legend=False,
    figsize=(10, 8)
)

plt.gca().invert_yaxis()
plt.title("Top 30 importâncias - XGBoost Tunado")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.savefig("feature_importance_xgboost_tunado.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Tempo total do experimento

In [ ]:
fim = time.perf_counter()

tempo_total = fim - inicio

horas = int(tempo_total // 3600)
minutos = int((tempo_total % 3600) // 60)
segundos = tempo_total % 60

print(f"Tempo total do experimento: {horas}h {minutos}min {segundos:.2f}s")